# Breast Cancer Prediction using Deep Learning

**Multi-Layer Perceptron (MLP) classifier for the Breast Cancer Wisconsin Diagnostic Dataset (WDBC)**

This notebook reproduces the full pipeline described in the accompanying project report:
data loading, preprocessing, model architecture, training, validation, evaluation
(accuracy, precision, recall, F1-score, confusion matrix, ROC curve), model saving, and
prediction examples.

Authors: SK Salma, P. Durgamma, Nithiasri S


## 1. Setup & Imports

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, auc, confusion_matrix, classification_report
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("TensorFlow version:", tf.__version__)


## 2. Data Loading

Loads `dataset/data.csv` if present, otherwise falls back to the identical dataset bundled with scikit-learn.

In [ ]:
DATA_PATH = "../dataset/data.csv"

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
else:
    print(f"'{DATA_PATH}' not found — using scikit-learn's bundled WDBC dataset instead.")
    from sklearn.datasets import load_breast_cancer
    data = load_breast_cancer(as_frame=True)
    df = data.frame.copy()
    df["diagnosis"] = df["target"].map({1: "B", 0: "M"})
    df = df.drop(columns=["target"])

print(df.shape)
df.head()


## 3. Exploratory Data Analysis

In [ ]:
df.info()


In [ ]:
df.describe().T


In [ ]:
# Class distribution (Figure 1 in the report)
plt.figure(figsize=(5,4))
ax = sns.countplot(x="diagnosis", data=df, palette=["#55A868", "#C44E52"])
ax.set_xticklabels(["Benign (B)", "Malignant (M)"])
plt.title("Distribution of Benign and Malignant Cases")
plt.ylabel("Count")
plt.xlabel("")
plt.tight_layout()
plt.savefig("../images/class_distribution.png", dpi=150)
plt.show()

print(df["diagnosis"].value_counts(normalize=True) * 100)


## 4. Data Preprocessing

- Drop irrelevant columns (`id`, `Unnamed: 32`)
- Label-encode `diagnosis`: B -> 0, M -> 1
- Min-Max scale all 30 numeric features to [0, 1]
- Stratified train/test split: 75% / 25%

In [ ]:
# Drop irrelevant columns
drop_cols = [c for c in ["id", "Unnamed: 32"] if c in df.columns]
df = df.drop(columns=drop_cols)
df = df.dropna(axis=1, how="all")

# Label encode diagnosis: B -> 0, M -> 1
encoder = LabelEncoder()
encoder.fit(["B", "M"])
df["diagnosis"] = encoder.transform(df["diagnosis"])

feature_names = [c for c in df.columns if c != "diagnosis"]
X = df[feature_names].values
y = df["diagnosis"].values

print("Feature matrix shape:", X.shape)
print("Label distribution:", np.bincount(y))


In [ ]:
# Stratified train/test split (75% / 25%), as per report section 4.1
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

# Min-Max Scaling: x_scaled = (x - x_min) / (x_max - x_min)  [Equation 1]
scaler = MinMaxScaler(feature_range=(0, 1))
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Training samples: {X_train.shape[0]} | Testing samples: {X_test.shape[0]}")


## 5. Model Architecture (Multi-Layer Perceptron)

| Layer | Neurons | Activation |
|---|---|---|
| Input | 30 | - |
| Hidden Layer 1 | 16 | ReLU |
| Dropout | - | p=0.3 |
| Hidden Layer 2 | 8 | ReLU |
| Dropout | - | p=0.3 |
| Output | 1 | Sigmoid |

In [ ]:
def build_mlp_model(input_dim=30, learning_rate=0.001):
    model = keras.Sequential(name="Breast_Cancer_MLP")
    model.add(keras.Input(shape=(input_dim,), name="input_features"))
    model.add(layers.Dense(16, activation="relu", name="hidden_layer_1"))
    model.add(layers.Dropout(0.3, name="dropout_1"))
    model.add(layers.Dense(8, activation="relu", name="hidden_layer_2"))
    model.add(layers.Dropout(0.3, name="dropout_2"))
    model.add(layers.Dense(1, activation="sigmoid", name="output"))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model

model = build_mlp_model(input_dim=X_train.shape[1])
model.summary()


## 6. Training

Adam optimizer, binary cross-entropy loss, batch size 32, 100 epochs, with EarlyStopping (patience=10, monitoring `val_accuracy`) and ModelCheckpoint.

In [ ]:
os.makedirs("../models", exist_ok=True)
os.makedirs("../images", exist_ok=True)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=10, restore_best_weights=True, mode="max", verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath="../models/trained_model.h5", monitor="val_accuracy",
        save_best_only=True, mode="max", verbose=0
    ),
]

history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=2,
)


In [ ]:
# Always save the final model explicitly as well
model.save("../models/trained_model.h5")
print("Model saved to ../models/trained_model.h5")


## 7. Training & Validation Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history["accuracy"], label="Training Accuracy", linewidth=2)
axes[0].plot(history.history["val_accuracy"], label="Validation Accuracy", linewidth=2)
axes[0].set_title("Training and Validation Accuracy")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy")
axes[0].legend(loc="lower right"); axes[0].grid(alpha=0.3)

axes[1].plot(history.history["loss"], label="Training Loss", linewidth=2)
axes[1].plot(history.history["val_loss"], label="Validation Loss", linewidth=2)
axes[1].set_title("Training and Validation Loss")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Binary Cross-Entropy Loss")
axes[1].legend(loc="upper right"); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../images/accuracy.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Test Set Evaluation

Accuracy, Precision, Recall, F1-score, ROC-AUC, and the Confusion Matrix.

In [ ]:
y_prob = model.predict(X_test).ravel()
y_pred = (y_prob >= 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc_value = roc_auc_score(y_test, y_prob)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc_value:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["Benign", "Malignant"]))


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Benign", "Malignant"], yticklabels=["Benign", "Malignant"], cbar=False)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.tight_layout()
plt.savefig("../images/confusion_matrix.png", dpi=150)
plt.show()


## 9. ROC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc_curve = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color="darkorange", linewidth=2, label=f"ROC curve (AUC = {roc_auc_curve:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="navy")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Receiver Operating Characteristic (ROC) Curve")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../images/roc_curve.png", dpi=150)
plt.show()


## 10. Prediction Examples

Run the trained model on a handful of held-out test samples to sanity-check individual predictions.

In [ ]:
sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_test), size=8, replace=False)

results = pd.DataFrame({
    "Actual": np.where(y_test[sample_idx] == 1, "Malignant", "Benign"),
    "Predicted": np.where(y_pred[sample_idx] == 1, "Malignant", "Benign"),
    "P(Malignant)": np.round(y_prob[sample_idx], 4),
})
results


## 11. Model Comparison (Extended Analysis)

As explored further in the project report (Chapter 6), the baseline MLP above was
compared against three additional models for a more thorough comparative analysis:

1. **Initial Neural Network** — a minimally-tuned baseline MLP.
2. **Tuned Neural Network** — the MLP above, enhanced with Batch Normalization and
   tuned hyperparameters/callbacks (this is the model built in Sections 5-9).
3. **Inception-like Neural Network** — parallel Dense branches concatenated together,
   inspired by the Inception architecture, to capture features at multiple levels of
   abstraction simultaneously.
4. **XGBoost Classifier** — a gradient-boosted decision tree ensemble, included as a
   strong non-neural baseline for tabular data.

The cell below trains all three additional models and produces the same suite of
metrics for direct comparison. This section is optional and can be skipped if you
only need the primary MLP model.


In [ ]:
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Concatenate
from tensorflow.keras.models import Model

# ---- (a) Initial Neural Network: minimal, untuned baseline ----
initial_model = keras.Sequential([
    keras.Input(shape=(X_train.shape[1],)),
    layers.Dense(4, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
initial_model.compile(optimizer="sgd", loss="binary_crossentropy", metrics=["accuracy"])
initial_history = initial_model.fit(
    X_train, y_train, validation_split=0.15, epochs=20, batch_size=32, verbose=0
)

# ---- (b) Tuned Neural Network: MLP + BatchNorm ----
tuned_model = keras.Sequential([
    keras.Input(shape=(X_train.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(16, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
])
tuned_model.compile(optimizer=keras.optimizers.Adam(0.001),
                     loss="binary_crossentropy", metrics=["accuracy"])
tuned_history = tuned_model.fit(
    X_train, y_train, validation_split=0.15, epochs=100, batch_size=32,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=10,
                                              restore_best_weights=True, mode="max")],
    verbose=0,
)

# ---- (c) Inception-like Neural Network ----
inp = Input(shape=(X_train.shape[1],))
branch_a = Dense(16, activation="relu")(inp)
branch_b = Dense(8, activation="relu")(inp)
branch_c = Dense(4, activation="relu")(inp)
merged = Concatenate()([branch_a, branch_b, branch_c])
x = Dense(16, activation="relu")(merged)
x = Dropout(0.3)(x)
out = Dense(1, activation="sigmoid")(x)
inception_model = Model(inputs=inp, outputs=out, name="Inception_like_Model")
inception_model.compile(optimizer=keras.optimizers.Adam(0.001),
                         loss="binary_crossentropy", metrics=["accuracy"])
inception_history = inception_model.fit(
    X_train, y_train, validation_split=0.15, epochs=100, batch_size=32,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=10,
                                              restore_best_weights=True, mode="max")],
    verbose=0,
)

print("All comparison models trained.")


In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    use_label_encoder=False, eval_metric="logloss", random_state=RANDOM_STATE,
)
xgb_model.fit(X_train, y_train)

def get_metrics(name, y_true, y_prob, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="weighted"),
        "Recall": recall_score(y_true, y_pred, average="weighted"),
        "F1-Score": f1_score(y_true, y_pred, average="weighted"),
        "AUC": roc_auc_score(y_true, y_prob),
    }

comparison_rows = []

p = initial_model.predict(X_test).ravel()
comparison_rows.append(get_metrics("Initial Neural Network", y_test, p, (p >= 0.5).astype(int)))

p = tuned_model.predict(X_test).ravel()
comparison_rows.append(get_metrics("Tuned Neural Network", y_test, p, (p >= 0.5).astype(int)))

p = inception_model.predict(X_test).ravel()
comparison_rows.append(get_metrics("Inception-like Neural Network", y_test, p, (p >= 0.5).astype(int)))

p = xgb_model.predict_proba(X_test)[:, 1]
comparison_rows.append(get_metrics("XGBoost Classifier", y_test, p, xgb_model.predict(X_test)))

comparison_df = pd.DataFrame(comparison_rows).set_index("Model")
comparison_df


In [ ]:
comparison_df[["Accuracy", "AUC"]].plot(kind="bar", figsize=(9, 5), colormap="viridis")
plt.title("Comparative Analysis of Model Performance")
plt.ylabel("Score")
plt.xticks(rotation=20, ha="right")
plt.ylim(0, 1.05)
plt.tight_layout()
plt.savefig("../images/model_comparison.png", dpi=150)
plt.show()


## 12. Conclusion

The primary MLP model (Sections 5-9) achieves strong performance in line with the project report (accuracy ≈ 97%, AUC ≈ 0.98), confirming that a lightweight, well-regularized feed-forward network is an effective and computationally efficient approach for tabular breast-cancer diagnosis classification.